# MetaEngine — Colab T4 GPU Worker (Autonomous)

**Runs MetaEngine benchmark directly on Colab T4 GPU — no Ray, no ngrok, no sandbox needed.**

## Features:
- Clones latest MetaEngine from GitHub (auto-pulls fixes)
- Uses T4 GPU for BoTorch GP fitting (10-50× faster)
- OpenRouter LLM judges (free Gemma + Nemotron models)
- Results sync to Turso cloud DB automatically
- `--no-cache` ensures each round runs fresh MetaEngine
- Runs infinitely (~12 hours per Colab session)

## Setup:
1. Set your API keys in Cell 3 (TURSO_DB_TOKEN, OPENROUTER_API_KEY)
2. Runtime → Change runtime type → T4 GPU
3. Runtime → Run all (Ctrl+F9)

## Multiple shards:
Open multiple Colab notebooks, change `--shard-id` in Cell 5:
- Notebook 1: `--shard-id 0 --instance-id colab-shard0`
- Notebook 2: `--shard-id 1 --instance-id colab-shard1`
- etc.

In [ ]:
# === CELL 1: Clone latest MetaEngine from GitHub ===
import os

if not os.path.exists('/content/EngineTest'):
    !git clone --depth 1 https://github.com/PatrickFrome/EngineTest.git /content/EngineTest
    print('✓ Repo cloned')
else:
    print('Repo exists, pulling latest...')
    
%cd /content/EngineTest
!git pull origin main 2>/dev/null || true
%cd METAENGINE_SLICE3_RESTORED
print('✓ Ready at:', os.getcwd())

In [ ]:
# === CELL 2: Install dependencies ===
!pip install -e . --no-deps -q 2>/dev/null
!pip install structlog litellm cryptography jsonschema -q 2>/dev/null
!pip install botorch -q 2>/dev/null || echo 'botorch optional (GPU GP fitting not available)'

# Verify
import structlog, litellm
print('✓ structlog + litellm installed')
try:
    import botorch
    print(f'✓ botorch {botorch.__version__} installed')
except:
    print('⚠ botorch not available (GP surrogate will use CPU heuristic)')

In [ ]:
# === CELL 3: Set API keys ===
# Paste your keys below (from .env.local or from the chat)

import os

# Turso cloud DB (for syncing results)
os.environ['TURSO_DB_TOKEN'] = 'YOUR_TURSO_TOKEN'
os.environ['TURSO_DB_HOST'] = 'metaengine-project-patrickfrome.aws-eu-west-1.turso.io'

# OpenRouter (free LLM judges — Gemma + Nemotron)
os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1-YOUR_OPENROUTER_KEY'

# Project root
os.environ['ME_BENCHMARK_ROOT'] = '/content/EngineTest/METAENGINE_SLICE3_RESTORED'

print('✓ API keys configured')
print(f'  Turso: {os.environ["TURSO_DB_HOST"][:30]}...')
print(f'  OpenRouter: {os.environ["OPENROUTER_API_KEY"][:15]}...')

In [ ]:
# === CELL 4: Clear old cache (force fresh runs) ===
import glob, os, shutil

cache_dir = '/content/EngineTest/METAENGINE_SLICE3_RESTORED/storage/result_cache'
if os.path.exists(cache_dir):
    files = glob.glob(cache_dir + '/*.json')
    for f in files:
        os.remove(f)
    print(f'✓ Cleared {len(files)} cached results')
else:
    print('✓ No cache dir (fresh start)')

# Also clear old benchmark task dirs
tasks_dir = '/content/EngineTest/METAENGINE_SLICE3_RESTORED/storage'
for item in os.listdir(tasks_dir):
    if item.startswith('massive_benchmark_tasks_'):
        shutil.rmtree(os.path.join(tasks_dir, item), ignore_errors=True)
        print(f'  cleared: {item}')

In [ ]:
# === CELL 5: Run benchmark (INFINITE, --no-cache, shard 0/8) ===
# 
# This cell runs FOREVER until Colab disconnects (~12 hours)
# Results sync to Turso cloud DB automatically after each round
# 
# To run a DIFFERENT shard, change:
#   --shard-id 1 --instance-id colab-shard1  (for shard 1/8)
#   --shard-id 2 --instance-id colab-shard2  (for shard 2/8)
#   etc.

!python3 scripts/run_massive_benchmark.py \
    --rounds 0 \
    --tasks-per-round 0 \
    --max-workers 2 \
    --no-zai \
    --minimal-output \
    --no-cache \
    --instance-id colab-gpu-shard0 \
    --shard-id 0 \
    --shard-count 8

## 📊 What happens:

1. **Round 1**: MetaEngine runs 14 tasks on T4 GPU (~15 seconds)
2. Results sync to Turso cloud DB
3. **Round 2**: Fresh run (no cache!) — results may differ
4. Repeats infinitely (~12 hours)

## 🔍 Monitor results:

Open another Colab cell and run:
```python
import json, urllib.request
req = urllib.request.Request(
    'https://metaengine-project-patrickfrome.aws-eu-west-1.turso.io/v2/pipeline',
    data=json.dumps({'requests':[{'type':'execute','stmt':{'sql':"SELECT count(*) FROM metaengine_artifacts WHERE artifact_kind='benchmark_task_result'"}}]}).encode(),
    headers={'Authorization':'Bearer YOUR_TURSO_TOKEN','Content-Type':'application/json'},
    method='POST'
)
d = json.loads(urllib.request.urlopen(req, timeout=15).read().decode())
print('Total results:', int(d['results'][0]['response']['result']['rows'][0][0]['value']))
```

## ⚠️ When Colab disconnects (~12h):
Just Runtime → Run all again. New session = new 12 hours.